In [1]:
import numpy as np

from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, accuracy_score, mean_squared_error

from numba import njit

import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots

import ipywidgets as widgets

# Normal With Discrete U

In [10]:
time = 1000 # Total seconds
dt = 0.0001
steps = int(time / dt) # Total steps
tau_steps = int(.7 / dt)  # How far we see back

test_size = .2

m = 1.0
k = 2.0
mu = 0.3

t = np.linspace(0, time, steps)
u = (t.astype(int) % 2) * 2 - 1 # Basially just cut down just the int of time no dec then mod to get 0 or 1 every sec

In [11]:
@njit
def run_simulation(steps, dt, m, k, mu, u):
    x = np.zeros(steps)
    v = np.zeros(steps)
    x[0] = 1.0
    v[0] = 0.0

    for i in range(1, steps):
        x_dot = v[i - 1]
        v_dot = -mu / m * v[i - 1] - k / m * x[i - 1] + u[i - 1] / m

        x[i] = x[i - 1] + x_dot * dt
        v[i] = v[i - 1] + v_dot * dt

    return x, v


x, v = run_simulation(steps, dt, m, k, mu, u)

In [12]:
X = np.column_stack((x, v))
X_train, X_test, y_train, y_test = train_test_split(
    X[tau_steps:], u[:-tau_steps], test_size=test_size, random_state=42
)

In [13]:
ridge_model = RidgeCV()
ridge_model.fit(X_train, y_train)
y_pred_ridge = ridge_model.predict(X_test)

In [14]:
y_pred_discrete = np.where(y_pred_ridge > 0.0, 1, -1)

r_2 = r2_score(y_test, y_pred_ridge)
accuracy = accuracy_score(y_test, y_pred_discrete) * 100

print(f"{r_2:.4f}", f"{accuracy:.2f}%")

0.7582 98.96%


In [15]:
num_points_plot = 50
t_plot = np.arange(num_points_plot)

# num_points_plot = 10000
# t_plot = np.arange(num_points_plot)

# start, end = 4000, 4500
# x_3d = x[start + tau_steps : end]
# v_3d = v[start + tau_steps : end]

# true_u_slice = u[start:end]
# pred_u_raw_slice = ridge_model.predict(X[start + tau_steps: num_points_plot + tau_steps])

# Slice out the first 50 shuffled points to mirror your previous plot
u_slice_plot = y_test[:num_points_plot]
u_pred_plot = y_pred_ridge[:num_points_plot]
u_pred_disc_plot = y_pred_discrete[:num_points_plot]

fig = go.Figure()

# Right points black circles
fig.add_trace(
    go.Scatter(
        x=t_plot,
        y=u_slice_plot,
        mode="markers",
        marker=dict(color="black", size=12),
        name="Correct Input",
    )
)

# Model guess
fig.add_trace(
    go.Scatter(
        x=t_plot,
        y=u_pred_plot,
        mode="markers",
        marker=dict(color="orange", symbol="x", size=10),
        name="Ridge Reg Guess",
    )
)

# Blue circle is the right just between -1 and 1
fig.add_trace(
    go.Scatter(
        x=t_plot,
        y=u_pred_disc_plot,
        mode="markers",
        marker=dict(color="rgba(0,0,0,0)", size=16, line=dict(color="blue", width=2)),
        name="Final Binary Decision",
    )
)

# Middle
fig.add_shape(
    type="line",
    x0=0,
    y0=0,
    x1=num_points_plot - 1,
    y1=0,
    line=dict(color="red", width=1.5, dash="dash"),
)

fig.update_layout(
    title=f"{num_points_plot} Random Points for True vs Predicted Input Force",
    xaxis_title="Index of Random Points",
    yaxis_title="Input Force Value (u)",
    yaxis=dict(range=[-1.5, 1.5]),  # Expanded range to fit -1 and 1 cleanly
    template="plotly_white",
    legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99),
    margin=dict(l=50, r=50, t=60, b=50),
)

fig.show()

In [8]:
w1, w2 = ridge_model.coef_[0], ridge_model.coef_[1]
b = ridge_model.intercept_

start, end = 4000, 4500
x_3d = x[start + tau_steps : end]
v_3d = v[start + tau_steps : end]
u_pred_3d = w1 * x_3d + w2 * v_3d + b


# Grid
x_range = np.linspace(x_3d.min() - 0.2, x_3d.max() + 0.2, 16)
v_range = np.linspace(v_3d.min() - 0.2, v_3d.max() + 0.2, 16)
X_grid, V_grid = np.meshgrid(x_range, v_range)


# Plane
U_plane = w1 * X_grid + w2 * V_grid + b

# Floor for green actual path and arrows
z_floor_val = U_plane.min() + 0.3
z_floor = np.full_like(u_pred_3d, z_floor_val)

# Vector
button_u_vals = [0, 1, -1]
arrow_traces = []
arrow_scale = 0.07
for i, u_val in enumerate(button_u_vals):
    dX_dt_grid = V_grid
    dV_dt_grid = (-mu / m) * V_grid - (k / m) * X_grid + u_val / m

    # Norm it
    grid_norm = np.sqrt(dX_dt_grid**2 + dV_dt_grid**2)
    grid_norm[grid_norm == 0] = 1.0
    dX_grid_static = (dX_dt_grid / grid_norm) * arrow_scale
    dV_grid_static = (dV_dt_grid / grid_norm) * arrow_scale

    temp_quiver = ff.create_quiver(
        X_grid, V_grid, dX_grid_static, dV_grid_static, scale=1, arrow_scale=0.3
    )
    temp_trace = temp_quiver.data[0]
    z_quiver_3d = [z_floor_val if val is not None else None for val in temp_trace.x]

    arrow_traces.append(
        go.Scatter3d(
            x=temp_trace.x,
            y=temp_trace.y,
            z=z_quiver_3d,
            mode="lines",
            line=dict(color="rgba(255, 0, 0, 0.3)", width=1),
            name=f"Physics Flow Field (u={u_val})",
            visible=i == 0,
        )
    )

fig = go.Figure()

# Surface
fig.add_trace(
    go.Surface(
        x=X_grid,
        y=V_grid,
        z=U_plane,
        colorscale="YlOrRd",
        opacity=0.35,
        showscale=False,
        name="Linear Regression Plane",
    )
)

# Blue Traj on prediction
fig.add_trace(
    go.Scatter3d(
        x=x_3d,
        y=v_3d,
        z=u_pred_3d,
        mode="lines",
        line=dict(color="blue", width=6),
        name="Active 3D Trajectory String",
    )
)

# Path taken
fig.add_trace(
    go.Scatter3d(
        x=x_3d,
        y=v_3d,
        z=z_floor,
        mode="lines",
        line=dict(color="rgba(0, 200, 100, 0.8)", width=5),
        name="Highlighted Floor Track (Footprints)",
    )
)

# Arrows
for trace in arrow_traces:
    fig.add_trace(trace)


buttons = []
# Basically the first 3 trace are always on
base_visibility = [True, True, True]
for i, u_val in enumerate(button_u_vals):
    # All traces are false except the one for the current u_val
    arrow_visibility = [False] * len(button_u_vals)
    arrow_visibility[i] = True

    buttons.append(
        dict(
            label=f"Field (u={u_val})",
            method="update",
            args=[{"visible": base_visibility + arrow_visibility}],
        )
    )

fig.update_layout(
    title="3D Phase Map: Background Field vs. Highlighted System Path",
    scene=dict(
        xaxis_title="Position (x)",
        yaxis_title="Velocity (v)",
        zaxis_title="Predicted Force (u)",
        camera=dict(eye=dict(x=1.4, y=1.4, z=1.2)),
    ),
    template="plotly_white",
    width=1000,
    height=800,
    updatemenus=[
        dict(
            type="buttons",
            direction="down",
            x=1.02,
            y=0.35,
            xanchor="left",
            yanchor="top",
            showactive=True,
            buttons=buttons,
        )
    ],
)

fig.show()

ValueError: zero-size array to reduction operation minimum which has no identity

In [ ]:
# Transient
w1, w2 = ridge_model.coef_[0], ridge_model.coef_[1]
b = ridge_model.intercept_

start, end = 0, 3000
x_3d = x[start + tau_steps : end]
v_3d = v[start + tau_steps : end]
u_pred_3d = w1 * x_3d + w2 * v_3d + b


# Grid
x_range = np.linspace(x_3d.min() - 0.2, x_3d.max() + 0.2, 16)
v_range = np.linspace(v_3d.min() - 0.2, v_3d.max() + 0.2, 16)
X_grid, V_grid = np.meshgrid(x_range, v_range)


# Plane
U_plane = w1 * X_grid + w2 * V_grid + b

# Floor for green actual path and arrows
z_floor_val = U_plane.min() + 0.3
z_floor = np.full_like(u_pred_3d, z_floor_val)

# Vector
button_u_vals = [0, 1, -1]
arrow_traces = []
arrow_scale = 0.07
for i, u_val in enumerate(button_u_vals):
    dX_dt_grid = V_grid
    dV_dt_grid = (-mu / m) * V_grid - (k / m) * X_grid + u_val / m

    # Norm it
    grid_norm = np.sqrt(dX_dt_grid**2 + dV_dt_grid**2)
    grid_norm[grid_norm == 0] = 1.0
    dX_grid_static = (dX_dt_grid / grid_norm) * arrow_scale
    dV_grid_static = (dV_dt_grid / grid_norm) * arrow_scale

    temp_quiver = ff.create_quiver(
        X_grid, V_grid, dX_grid_static, dV_grid_static, scale=1, arrow_scale=0.3
    )
    temp_trace = temp_quiver.data[0]
    z_quiver_3d = [z_floor_val if val is not None else None for val in temp_trace.x]

    arrow_traces.append(
        go.Scatter3d(
            x=temp_trace.x,
            y=temp_trace.y,
            z=z_quiver_3d,
            mode="lines",
            line=dict(color="rgba(255, 0, 0, 0.3)", width=1),
            name=f"Physics Flow Field (u={u_val})",
            visible=i == 0,
        )
    )

fig = go.Figure()

# Surface
fig.add_trace(
    go.Surface(
        x=X_grid,
        y=V_grid,
        z=U_plane,
        colorscale="YlOrRd",
        opacity=0.35,
        showscale=False,
        name="Linear Regression Plane",
    )
)

# Blue Traj on prediction
fig.add_trace(
    go.Scatter3d(
        x=x_3d,
        y=v_3d,
        z=u_pred_3d,
        mode="lines",
        line=dict(color="blue", width=6),
        name="Active 3D Trajectory String",
    )
)

# Path taken
fig.add_trace(
    go.Scatter3d(
        x=x_3d,
        y=v_3d,
        z=z_floor,
        mode="lines",
        line=dict(color="rgba(0, 200, 100, 0.8)", width=5),
        name="Highlighted Floor Track (Footprints)",
    )
)

# Arrows
for trace in arrow_traces:
    fig.add_trace(trace)


buttons = []
# Basically the first 3 trace are always on
base_visibility = [True, True, True]
for i, u_val in enumerate(button_u_vals):
    # All traces are false except the one for the current u_val
    arrow_visibility = [False] * len(button_u_vals)
    arrow_visibility[i] = True

    buttons.append(
        dict(
            label=f"Field (u={u_val})",
            method="update",
            args=[{"visible": base_visibility + arrow_visibility}],
        )
    )

fig.update_layout(
    title="3D Phase Map: Background Field vs. Highlighted System Path",
    scene=dict(
        xaxis_title="Position (x)",
        yaxis_title="Velocity (v)",
        zaxis_title="Predicted Force (u)",
        camera=dict(eye=dict(x=1.4, y=1.4, z=1.2)),
    ),
    template="plotly_white",
    width=1000,
    height=800,
    updatemenus=[
        dict(
            type="buttons",
            direction="down",
            x=1.02,
            y=0.35,
            xanchor="left",
            yanchor="top",
            showactive=True,
            buttons=buttons,
        )
    ],
)

fig.show()

# U as Sin Wave

In [55]:
time = 1000  # Total seconds
dt = 0.01
steps = int(time / dt)  # Total steps
tau_steps = int(0.7 / dt)  # How far we see back

test_size = 0.2

m = 1.0
k = 2.0
mu = 0.3

t = np.linspace(0, time, steps)
u = np.sin(t)

In [56]:
@njit
def run_simulation(steps, dt, m, k, mu, u):
    x = np.zeros(steps)
    v = np.zeros(steps)
    x[0] = 0
    v[0] = 0.0

    for i in range(1, steps):
        x_dot = v[i - 1]
        v_dot = -mu / m * v[i - 1] - k / m * x[i - 1] + u[i - 1] / m

        x[i] = x[i - 1] + x_dot * dt
        v[i] = v[i - 1] + v_dot * dt

    return x, v


x, v = run_simulation(steps, dt, m, k, mu, u)

In [57]:
X = np.column_stack((x, v))
X_train, X_test, y_train, y_test = train_test_split(
    X[tau_steps:], u[:-tau_steps], test_size=test_size, random_state=42
)

In [58]:
ridge_model = RidgeCV()
ridge_model.fit(X_train, y_train)
y_pred_ridge = ridge_model.predict(X_test)

In [59]:
r_2 = r2_score(y_test, y_pred_ridge)
mse = mean_squared_error(y_test, y_pred_ridge)

print(f"{r_2:.4f}", f"{mse:.4f}")

0.9982 0.0009


In [12]:
num_points_plot = 10000
t_plot = np.arange(num_points_plot)

true_u_slice = u[:num_points_plot]
pred_u_raw_slice = ridge_model.predict(X[tau_steps:num_points_plot + tau_steps])

fig = go.Figure()

# Right points black circles
fig.add_trace(
    go.Scatter(
        x=t_plot,
        y=true_u_slice,
        mode="markers",
        marker=dict(color="black", size=12),
        name="Correct Input",
    )
)

# Model guess
fig.add_trace(
    go.Scatter(
        x=t_plot,
        y=pred_u_raw_slice,
        mode="markers",
        marker=dict(color="orange", symbol="x", size=10),
        name="Ridge Reg Guess",
    )
)

fig.update_layout(
    title=f"{num_points_plot} Random Points for True vs Predicted Input Force",
    xaxis_title="Index of Random Points",
    yaxis_title="Input Force Value (u)",
    yaxis=dict(range=[-1.5, 1.5]),
    template="plotly_white",
    legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99),
    margin=dict(l=50, r=50, t=60, b=50),
)

fig.show()

In [16]:
w1, w2 = ridge_model.coef_[0], ridge_model.coef_[1]
b = ridge_model.intercept_

start, end = 4000, 5000
x_3d = x[start + tau_steps : end]
v_3d = v[start + tau_steps : end]
u_pred_3d = w1 * x_3d + w2 * v_3d + b


# Grid
x_range = np.linspace(x_3d.min() - 0.2, x_3d.max() + 0.2, 16)
v_range = np.linspace(v_3d.min() - 0.2, v_3d.max() + 0.2, 16)
X_grid, V_grid = np.meshgrid(x_range, v_range)


# Plane
U_plane = w1 * X_grid + w2 * V_grid + b

# Floor for green actual path and arrows
z_floor_val = U_plane.min() + 0.3
z_floor = np.full_like(u_pred_3d, z_floor_val)

# Vector
num_steps = 9
t_vals = np.linspace(0, 2 * np.pi, num_steps)
slider_u_vals = np.sin(t_vals)
arrow_traces = []
arrow_scale = 0.07
for i, u_val in enumerate(slider_u_vals):
    dX_dt_grid = V_grid
    dV_dt_grid = (-mu / m) * V_grid - (k / m) * X_grid + u_val / m

    # Norm it
    grid_norm = np.sqrt(dX_dt_grid**2 + dV_dt_grid**2)
    grid_norm[grid_norm == 0] = 1.0
    dX_grid_static = (dX_dt_grid / grid_norm) * arrow_scale
    dV_grid_static = (dV_dt_grid / grid_norm) * arrow_scale

    temp_quiver = ff.create_quiver(
        X_grid, V_grid, dX_grid_static, dV_grid_static, scale=1, arrow_scale=0.3
    )
    temp_trace = temp_quiver.data[0]
    z_quiver_3d = [z_floor_val if val is not None else None for val in temp_trace.x]

    arrow_traces.append(
        go.Scatter3d(
            x=temp_trace.x,
            y=temp_trace.y,
            z=z_quiver_3d,
            mode="lines",
            line=dict(color="rgba(255, 0, 0, 0.3)", width=1),
            name=f"Physics Flow Field (u={u_val})",
            visible=i == 0,
        )
    )

fig = go.Figure()

# Surface
fig.add_trace(
    go.Surface(
        x=X_grid,
        y=V_grid,
        z=U_plane,
        colorscale="YlOrRd",
        opacity=0.35,
        showscale=False,
        name="Linear Regression Plane",
    )
)

# Blue Traj on prediction
fig.add_trace(
    go.Scatter3d(
        x=x_3d,
        y=v_3d,
        z=u_pred_3d,
        mode="lines",
        line=dict(color="blue", width=6),
        name="Active 3D Trajectory String",
    )
)

# Path taken
fig.add_trace(
    go.Scatter3d(
        x=x_3d,
        y=v_3d,
        z=z_floor,
        mode="lines",
        line=dict(color="rgba(0, 200, 100, 0.8)", width=5),
        name="Highlighted Floor Track (Footprints)",
    )
)

# Arrows
for trace in arrow_traces:
    fig.add_trace(trace)


slider_steps = []
# Basically the first 3 trace are always on
base_visibility = [True, True, True]
for i, t_val in enumerate(t_vals):
    # All traces are false except the one for the current u_val
    arrow_visibility = [False] * len(slider_u_vals)
    arrow_visibility[i] = True

    deg = int(np.degrees(t_val))
    u_inst = slider_u_vals[i]

    slider_steps.append(
        dict(
            label=f"{deg}° (t={t_val:.1f}s)",
            method="update",
            args=[{"visible": base_visibility + arrow_visibility}],
        )
    )

fig.update_layout(
    title="3D Phase Map: Background Field vs. Highlighted System Path",
    scene=dict(
        xaxis_title="Position (x)",
        yaxis_title="Velocity (v)",
        zaxis_title="Predicted Force (u)",
        camera=dict(eye=dict(x=1.4, y=1.4, z=1.2)),
    ),
    template="plotly_white",
    width=1000,
    height=900,
    sliders=[
        dict(
            active=0,
            currentvalue={
                "prefix": "Sine Input Forcing Frame: ",
                "font": {"size": 14, "color": "darkred"},
            },
            pad={"t": 50},
            steps=slider_steps,
        )
    ],
)

fig.show()

# Heatmaps

In [16]:
time = 500
dt = 0.01
steps = int(time / dt)
test_size = 0.2

m = 1.0
mu = 0.3

t = np.linspace(0, time, steps)
u = (t.astype(int) % 2) * 2 - 1


trial_count = 50
k_values = np.linspace(0.1, 20.0, trial_count)
tau_values = np.linspace(0.1, 3, trial_count)

# All combos of these together, Basically 100 tau for every k value
K_grid, TAU_grid = np.meshgrid(k_values, tau_values)
k_vector = K_grid.flatten()
tau_vector = TAU_grid.flatten()
num_trials = len(k_vector)

In [17]:
@njit
def run_simulation(steps, dt, m, mu, u, k_vector):
    num_trials = len(k_vector)

    X = np.ones((steps, num_trials))
    V = np.zeros((steps, num_trials))

    for i in range(1, steps):
        x_dot = V[i - 1]
        v_dot = -mu / m * V[i - 1] - k_vector / m * X[i - 1] + u[i - 1] / m

        X[i] = X[i - 1] + x_dot * dt
        V[i] = V[i - 1] + v_dot * dt

    return X, V


X, V = run_simulation(steps, dt, m, mu, u, k_vector)

In [18]:
r2_grid = np.zeros(num_trials)

for i in range(num_trials):
    tau_steps = int(tau_vector[i] / dt)

    x_trial = X[:, i]
    v_trial = V[:, i]
    X_mat = np.column_stack((x_trial, v_trial))

    X_train, X_test, y_train, y_test = train_test_split(
        X_mat[tau_steps:], u[:-tau_steps], test_size=test_size, random_state=42
    )

    ridge_model = RidgeCV()
    ridge_model.fit(X_train, y_train)
    y_pred_ridge = ridge_model.predict(X_test)

    r2_grid[i] = r2_score(y_test, y_pred_ridge)

In [19]:
fig = go.Figure(
    data=go.Heatmap(
        z=r2_grid.reshape(trial_count, trial_count),
        x=k_values,
        y=tau_values,
        colorscale="Viridis",
        hovertemplate="Stiffness (k): %{x:.2f}<br>Tau (periods): %{y:.2f}<br>R² Score: %{z:.4f}<extra></extra>",
        colorbar=dict(title="R² Score"),
    )
)

fig.update_layout(
    title=dict(
        text="Grid Search: Stiffness (k) vs. Lookback Window (τ)",
        x=0.5,
    ),
    xaxis=dict(title="Stiffness Coefficient (k)"),
    yaxis=dict(title="Tau Window (Seconds)"),
    width=700,
    height=600,
)

fig.show()

In [23]:
time = 500
dt = 0.01
steps = int(time / dt)
test_size = 0.2

m = 1.0
k = 10.0

t = np.linspace(0, time, steps)
u = (t.astype(int) % 2) * 2 - 1

trial_count = 50
mu_values = np.linspace(0.1, 20.0, trial_count)
tau_values = np.linspace(0.1, 3, trial_count)

# All combos of these together, Basically 100 tau for every k value
MU_grid, TAU_grid = np.meshgrid(mu_values, tau_values)
mu_vector = MU_grid.flatten()
tau_vector = TAU_grid.flatten()
num_trials = len(mu_vector)

In [24]:
@njit
def run_simulation(steps, dt, m, mu_vector, u, k):
    num_trials = len(mu_vector)

    X = np.ones((steps, num_trials))
    V = np.zeros((steps, num_trials))

    for i in range(1, steps):
        x_dot = V[i - 1]
        v_dot = -mu_vector / m * V[i - 1] - k / m * X[i - 1] + u[i - 1] / m

        X[i] = X[i - 1] + x_dot * dt
        V[i] = V[i - 1] + v_dot * dt

    return X, V


X, V = run_simulation(steps, dt, m, mu_vector, u, k)

In [25]:
r2_grid = np.zeros(num_trials)

for i in range(num_trials):
    tau_steps = int(tau_vector[i] / dt)

    x_trial = X[:, i]
    v_trial = V[:, i]
    X_mat = np.column_stack((x_trial, v_trial))

    X_train, X_test, y_train, y_test = train_test_split(
        X_mat[tau_steps:], u[:-tau_steps], test_size=test_size, random_state=42
    )

    ridge_model = RidgeCV()
    ridge_model.fit(X_train, y_train)
    y_pred_ridge = ridge_model.predict(X_test)

    y_pred_discrete = np.where(y_pred_ridge > 0.0, 1, -1)

    r2_grid[i] = r2_score(y_test, y_pred_ridge)

In [26]:
fig = go.Figure(
    data=go.Heatmap(
        z=r2_grid.reshape(trial_count, trial_count),
        x=mu_values,
        y=tau_values,
        colorscale="Viridis",
        hovertemplate="Mu (μ): %{x:.2f}<br>Tau (periods): %{y:.2f}<br>R² Score: %{z:.4f}<extra></extra>",
        colorbar=dict(
            title="R² Score"
        ),
    )
)

fig.update_layout(
    title=dict(
        text="Grid Search: Mu (μ) vs. Lookback Window (τ)",
        x=0.5,
    ),
    xaxis=dict(title="Mu (μ)"),
    yaxis=dict(title="Tau Window (Seconds)"),
    width=700,
    height=600,
)

fig.show()

In [21]:
time = 100
dt = 0.01
steps = int(time / dt)
test_size = 0.2

m = 1.0
mu = 0.3

t = np.linspace(0, time, steps)
u = np.sin((2 * np.pi) * t)


trial_count = 50
k_values = np.linspace(0.1, 20.0, trial_count)
tau_values = np.linspace(0.1, 3, trial_count)

# All combos of these together, Basically 100 tau for every k value
K_grid, TAU_grid = np.meshgrid(k_values, tau_values)
k_vector = K_grid.flatten()
tau_vector = TAU_grid.flatten()
num_trials = len(k_vector)

In [22]:
@njit
def run_simulation(steps, dt, m, mu, u, k_vector):
    num_trials = len(k_vector)

    # Pre-allocate history matrices: (steps, trials)
    X = np.ones((steps, num_trials))
    V = np.zeros((steps, num_trials))

    for i in range(1, steps):
        x_dot = V[i - 1]
        v_dot = -mu / m * V[i - 1] - k_vector / m * X[i - 1] + u[i - 1] / m

        X[i] = X[i - 1] + x_dot * dt
        V[i] = V[i - 1] + v_dot * dt

    return X, V


X, V = run_simulation(steps, dt, m, mu, u, k_vector)

In [23]:
r2_grid = np.zeros(num_trials)

for i in range(num_trials):
    # Calculate the specific time delay index for this specific trial
    tau_steps = int(tau_vector[i] / dt)

    # Extract x and v histories for this single trial out of the giant matrix
    x_trial = X[:, i]
    v_trial = V[:, i]
    X_mat = np.column_stack((x_trial, v_trial))

    # Your model training and prediction logic
    X_train, X_test, y_train, y_test = train_test_split(
        X_mat[tau_steps:], u[:-tau_steps], test_size=test_size, random_state=42
    )

    ridge_model = RidgeCV()
    ridge_model.fit(X_train, y_train)
    y_pred_ridge = ridge_model.predict(X_test)

    y_pred_discrete = np.where(y_pred_ridge > 0.0, 1, -1)

    # Store metrics into flattened lists
    r2_grid[i] = r2_score(y_test, y_pred_ridge)

In [24]:
fig = go.Figure(
    data=go.Heatmap(
        z=r2_grid.reshape(trial_count, trial_count),
        x=k_values,
        y=tau_values,
        colorscale="Viridis",
        hovertemplate="Stiffness (k): %{x:.2f}<br>Tau (periods): %{y:.2f}<br>R² Score: %{z:.4f}<extra></extra>",
        colorbar=dict(
            title="R² Score"
        ), 
    )
)

fig.update_layout(
    title=dict(
        text="Grid Search: Stiffness (k) vs. Lookback Window (τ)",
        x=0.5,
    ),
    xaxis=dict(title="Stiffness Coefficient (k)"),
    yaxis=dict(title="Tau Window (Seconds)"),
    width=700,
    height=600,
)

fig.show()

In [17]:
time = 100
dt = 0.01
steps = int(time / dt)
test_size = 0.2

m = 1.0
k = 1.0

t = np.linspace(0, time, steps)
u = np.sin(2 * np.pi * t)

trial_count = 50
mu_values = np.linspace(0.1, 20.0, trial_count)
tau_values = np.linspace(0.1, 3, trial_count)

# All combos of these together, Basically 100 tau for every k value
MU_grid, TAU_grid = np.meshgrid(mu_values, tau_values)
mu_vector = MU_grid.flatten()
tau_vector = TAU_grid.flatten()
num_trials = len(mu_vector)

In [18]:
@njit
def run_simulation(steps, dt, m, mu_vector, u, k):
    num_trials = len(mu_vector)

    X = np.ones((steps, num_trials))
    V = np.zeros((steps, num_trials))

    for i in range(1, steps):
        x_dot = V[i - 1]
        v_dot = -mu_vector / m * V[i - 1] - k / m * X[i - 1] + u[i - 1] / m

        X[i] = X[i - 1] + x_dot * dt
        V[i] = V[i - 1] + v_dot * dt

    return X, V


X, V = run_simulation(steps, dt, m, mu_vector, u, k)

In [19]:
r2_grid = np.zeros(num_trials)

for i in range(num_trials):
    tau_steps = int(tau_vector[i] / dt)

    x_trial = X[:, i]
    v_trial = V[:, i]
    X_mat = np.column_stack((x_trial, v_trial))

    X_train, X_test, y_train, y_test = train_test_split(
        X_mat[tau_steps:], u[:-tau_steps], test_size=test_size, random_state=42
    )

    ridge_model = RidgeCV()
    ridge_model.fit(X_train, y_train)
    y_pred_ridge = ridge_model.predict(X_test)

    r2_grid[i] = r2_score(y_test, y_pred_ridge)

In [20]:
fig = go.Figure(
    data=go.Heatmap(
        z=r2_grid.reshape(trial_count, trial_count),
        x=mu_values,
        y=tau_values,
        colorscale="Viridis",
        hovertemplate="Mu (μ): %{x:.2f}<br>Tau (seconds): %{y:.2f}<br>R² Score: %{z:.4f}<extra></extra>",
        colorbar=dict(title="R² Score"),
    )
)

fig.update_layout(
    title=dict(
        text="Grid Search: Mu (μ) vs. Lookback Window (τ)",
        x=0.5,
    ),
    xaxis=dict(title="Mu (μ)"),
    yaxis=dict(title="Tau Window (Seconds)"),
    width=700,
    height=600,
)

fig.show()

# More Pots?

In [17]:
time = 1000  # Total seconds
dt = 0.01
steps = int(time / dt)  # Total steps
tau_steps = int(.5 / dt)  # How far we see back

test_size = 0.2

m = 1.0
k = 2.0
mu = 20.0

t = np.linspace(0, time, steps)
u = (
    t.astype(int) % 2
) * 2 - 1  # Basially just cut down just the int of time no dec then mod to get 0 or 1 every sec


@njit
def run_simulation(steps, dt, m, k, mu, u):
    x = np.zeros(steps)
    v = np.zeros(steps)
    x[0] = 1.0
    v[0] = 0.0

    for i in range(1, steps):
        x_dot = v[i - 1]
        v_dot = -mu / m * v[i - 1] - k / m * x[i - 1] + u[i - 1] / m

        x[i] = x[i - 1] + x_dot * dt
        v[i] = v[i - 1] + v_dot * dt

    return x, v


x, v = run_simulation(steps, dt, m, k, mu, u)

X = np.column_stack((x, v))
X_train, X_test, y_train, y_test = train_test_split(
    X[tau_steps:], u[:-tau_steps], test_size=test_size, random_state=42
)

ridge_model = RidgeCV()
ridge_model.fit(X_train, y_train)
y_pred_ridge = ridge_model.predict(X_test)

y_pred_discrete = np.where(y_pred_ridge > 0.0, 1, -1)
r_2 = r2_score(y_test, y_pred_ridge)
accuracy = accuracy_score(y_test, y_pred_discrete) * 100
print(f"{r_2:.4f}", f"{accuracy:.2f}%")

0.0481 57.39%


In [18]:
w1, w2 = ridge_model.coef_[0], ridge_model.coef_[1]
b = ridge_model.intercept_

start, end = 4000, 4500
x_3d = x[start + tau_steps : end]
v_3d = v[start + tau_steps : end]
u_pred_3d = w1 * x_3d + w2 * v_3d + b


# Grid
x_range = np.linspace(x_3d.min() - 0.2, x_3d.max() + 0.2, 16)
v_range = np.linspace(v_3d.min() - 0.2, v_3d.max() + 0.2, 16)
X_grid, V_grid = np.meshgrid(x_range, v_range)


# Plane
U_plane = w1 * X_grid + w2 * V_grid + b

# Floor for green actual path and arrows
z_floor_val = U_plane.min() + 0.3
z_floor = np.full_like(u_pred_3d, z_floor_val)

# Vector
button_u_vals = [0, 1, -1]
arrow_traces = []
arrow_scale = 0.07
for i, u_val in enumerate(button_u_vals):
    dX_dt_grid = V_grid
    dV_dt_grid = (-mu / m) * V_grid - (k / m) * X_grid + u_val / m

    # Norm it
    grid_norm = np.sqrt(dX_dt_grid**2 + dV_dt_grid**2)
    grid_norm[grid_norm == 0] = 1.0
    dX_grid_static = (dX_dt_grid / grid_norm) * arrow_scale
    dV_grid_static = (dV_dt_grid / grid_norm) * arrow_scale

    temp_quiver = ff.create_quiver(
        X_grid, V_grid, dX_grid_static, dV_grid_static, scale=1, arrow_scale=0.3
    )
    temp_trace = temp_quiver.data[0]
    z_quiver_3d = [z_floor_val if val is not None else None for val in temp_trace.x]

    arrow_traces.append(
        go.Scatter3d(
            x=temp_trace.x,
            y=temp_trace.y,
            z=z_quiver_3d,
            mode="lines",
            line=dict(color="rgba(255, 0, 0, 0.3)", width=1),
            name=f"Physics Flow Field (u={u_val})",
            visible=i == 0,
        )
    )

fig = go.Figure()

# Surface
fig.add_trace(
    go.Surface(
        x=X_grid,
        y=V_grid,
        z=U_plane,
        colorscale="YlOrRd",
        opacity=0.35,
        showscale=False,
        name="Linear Regression Plane",
    )
)

# Blue Traj on prediction
fig.add_trace(
    go.Scatter3d(
        x=x_3d,
        y=v_3d,
        z=u_pred_3d,
        mode="lines",
        line=dict(color="blue", width=6),
        name="Active 3D Trajectory String",
    )
)

# Path taken
fig.add_trace(
    go.Scatter3d(
        x=x_3d,
        y=v_3d,
        z=z_floor,
        mode="lines",
        line=dict(color="rgba(0, 200, 100, 0.8)", width=5),
        name="Highlighted Floor Track (Footprints)",
    )
)

# Arrows
for trace in arrow_traces:
    fig.add_trace(trace)


buttons = []
# Basically the first 3 trace are always on
base_visibility = [True, True, True]
for i, u_val in enumerate(button_u_vals):
    # All traces are false except the one for the current u_val
    arrow_visibility = [False] * len(button_u_vals)
    arrow_visibility[i] = True

    buttons.append(
        dict(
            label=f"Field (u={u_val})",
            method="update",
            args=[{"visible": base_visibility + arrow_visibility}],
        )
    )

fig.update_layout(
    title="3D Phase Map: Background Field vs. Highlighted System Path",
    scene=dict(
        xaxis_title="Position (x)",
        yaxis_title="Velocity (v)",
        zaxis_title="Predicted Force (u)",
        camera=dict(eye=dict(x=1.4, y=1.4, z=1.2)),
    ),
    template="plotly_white",
    width=1000,
    height=800,
    updatemenus=[
        dict(
            type="buttons",
            direction="down",
            x=1.02,
            y=0.35,
            xanchor="left",
            yanchor="top",
            showactive=True,
            buttons=buttons,
        )
    ],
)

fig.show()

In [25]:
time = 1000
dt = 0.01
steps = int(time / dt)
test_size = 0.2
m = 1.0
k = 2.0

t = np.linspace(0, time, steps)
u = (t.astype(int) % 2) * 2 - 1

grid_size = (4, 4)
mu_values = np.linspace(1, 20, grid_size[0])
tau_values = np.linspace(0.5, 2.5, grid_size[1])

start, end = 4000, 4500


@njit
def run_simulation(steps, dt, m, k, mu, u):
    x = np.zeros(steps)
    v = np.zeros(steps)
    x[0] = 1.0
    v[0] = 0.0
    for i in range(1, steps):
        x_dot = v[i - 1]
        v_dot = -mu / m * v[i - 1] - k / m * x[i - 1] + u[i - 1] / m
        x[i] = x[i - 1] + x_dot * dt
        v[i] = v[i - 1] + v_dot * dt
    return x, v


#  Get data and names
titles = []
plot_data = {}

for row_idx, mu_val in enumerate(mu_values):
    x, v = run_simulation(steps, dt, m, k, mu_val, u)
    X = np.column_stack((x, v))

    for col_idx, tau_val in enumerate(tau_values):
        tau_steps = int(tau_val / dt)

        X_train, X_test, y_train, y_test = train_test_split(
            X[tau_steps:], u[:-tau_steps], test_size=test_size, random_state=42
        )

        ridge_model = RidgeCV()
        ridge_model.fit(X_train, y_train)
        y_pred_ridge = ridge_model.predict(X_test)
        y_pred_discrete = np.where(y_pred_ridge > 0.0, 1, -1)

        r2 = r2_score(y_test, y_pred_ridge)
        acc = accuracy_score(y_test, y_pred_discrete) * 100

        title_str = f"μ={mu_val:.1f}, τ={tau_val:.2f}<br>R²: {r2:.2f} | Acc: {acc:.1f}%"
        titles.append(title_str)

        plot_data[(row_idx, col_idx)] = (x, v, ridge_model, tau_steps)


fig = make_subplots(
    rows=4,
    cols=4,
    specs=[[{"type": "scene"} for _ in range(4)] for _ in range(4)],
    subplot_titles=titles,
    vertical_spacing=0.08,
)

for row_idx, mu_val in enumerate(mu_values):
    for col_idx, tau_val in enumerate(tau_values):
        x, v, ridge_model, tau_steps = plot_data[(row_idx, col_idx)]

        w1, w2 = ridge_model.coef_[0], ridge_model.coef_[1]
        b = ridge_model.intercept_

        x_3d = x[start + tau_steps : end]
        v_3d = v[start + tau_steps : end]
        u_pred_3d = w1 * x_3d + w2 * v_3d + b

        u_actual_3d = u[start : end - tau_steps]

        min_len = min(len(x_3d), len(u_actual_3d))
        x_3d = x_3d[:min_len]
        v_3d = v_3d[:min_len]
        u_pred_3d = u_pred_3d[:min_len]
        u_actual_3d = u_actual_3d[:min_len]

        x_range = np.linspace(x_3d.min() - 0.2, x_3d.max() + 0.2, 12)
        v_range = np.linspace(v_3d.min() - 0.2, v_3d.max() + 0.2, 12)
        X_grid, V_grid = np.meshgrid(x_range, v_range)
        U_plane = w1 * X_grid + w2 * V_grid + b

        z_floor_val = U_plane.min() - 0.2
        z_floor = np.full_like(u_pred_3d, z_floor_val)

        r, c = row_idx + 1, col_idx + 1

        # plane
        fig.add_trace(
            go.Surface(
                x=X_grid,
                y=V_grid,
                z=U_plane,
                colorscale="YlOrRd",
                opacity=0.25,
                showscale=False,
                showlegend=False,
            ),
            row=r,
            col=c,
        )

        # blue projection
        fig.add_trace(
            go.Scatter3d(
                x=x_3d,
                y=v_3d,
                z=u_pred_3d,
                mode="lines",
                line=dict(color="blue", width=4),
                showlegend=False,
            ),
            row=r,
            col=c,
        )

        # green base
        fig.add_trace(
            go.Scatter3d(
                x=x_3d,
                y=v_3d,
                z=z_floor,
                mode="lines",
                line=dict(color="rgba(0, 200, 100, 0.6)", width=3),
                showlegend=False,
            ),
            row=r,
            col=c,
        )

        # the target vals
        fig.add_trace(
            go.Scatter3d(
                x=x_3d,
                y=v_3d,
                z=u_actual_3d,
                mode="markers",
                marker=dict(
                    size=2.5,
                    color=u_actual_3d,
                    colorscale="Blackbody",  # Deep, high-contrast spectrum split
                    opacity=0.7,
                ),
                showlegend=False,
            ),
            row=r,
            col=c,
        )

update_dict = {}
for i in range(1, 17):
    scene_key = f"scene{i}" if i > 1 else "scene"
    update_dict[scene_key] = dict(
        xaxis=dict(title="", showticklabels=False),
        yaxis=dict(title="", showticklabels=False),
        zaxis=dict(title="", showticklabels=False),
        camera=dict(eye=dict(x=1.3, y=1.3, z=1.1)),
    )

fig.update_layout(
    title=dict(
        text="Reservoir Phase Maps: True Targets vs Predictions Across (μ, τ)",
        x=0.5,
    ),
    template="plotly_white",
    width=1300,
    height=1200,
    **update_dict,
)

fig.show()

In [16]:
time = 500
dt = 0.01
steps = int(time / dt)
test_size = 0.2
m = 1.0
start, end = 4000, 4500

t = np.linspace(0, time, steps)
u = (t.astype(int) % 2) * 2 - 1


@njit
def run_simulation(steps, dt, m, k, mu, u):
    x, v = np.zeros(steps), np.zeros(steps)
    x[0], v[0] = 1.0, 0.0
    for i in range(1, steps):
        x_dot = v[i - 1]
        v_dot = -mu / m * v[i - 1] - k / m * x[i - 1] + u[i - 1] / m
        x[i] = x[i - 1] + x_dot * dt
        v[i] = v[i - 1] + v_dot * dt
    return x, v


fig = go.FigureWidget()  # We use this to update

fig.add_trace(
    go.Surface(colorscale="YlOrRd", opacity=0.15, showscale=False, name="Plane")
)
fig.add_trace(
    go.Scatter3d(mode="lines", line=dict(color="blue", width=6), name="Prediction")
)
fig.add_trace(
    go.Scatter3d(
        mode="lines+markers",
        marker=dict(size=3.5, colorscale="Cividis"),
        line=dict(color="rgba(100,100,100,0.3)", width=2, dash="dash"),
        name="Target",
    )
)
fig.add_trace(
    go.Scatter3d(
        mode="markers",
        marker=dict(size=4.5, colorscale="Electric"),
        name="Drive (τ=0)",
    )
)
fig.update_layout(
    template="plotly_white",
    width=900,
    height=700,
    margin=dict(l=0, r=0, b=0, t=40),
    scene=dict(
        xaxis_title="Position (x)",
        yaxis_title="Velocity (v)",
        zaxis_title="Force Scales (u)",
        camera=dict(eye=dict(x=1.4, y=1.4, z=1.2)),
    ),
)


def update_plot(mu, tau, k):
    tau_steps = int(tau / dt)

    # sim
    x, v = run_simulation(steps, dt, m, k, mu, u)
    X = np.column_stack((x, v))

    # model
    effective_tau = max(1, tau_steps)
    X_train, X_test, y_train, y_test = train_test_split(
        X[effective_tau:],
        u[:-effective_tau],
        test_size=test_size,
        random_state=42,
    )
    ridge_model = RidgeCV().fit(X_train, y_train)

    w1, w2 = ridge_model.coef_[0], ridge_model.coef_[1]
    b = ridge_model.intercept_

    # metric
    y_pred = ridge_model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    acc = accuracy_score(y_test, np.where(y_pred > 0.0, 1, -1)) * 100

    # where we slice time
    x_3d = x[start + effective_tau : end]
    v_3d = v[start + effective_tau : end]
    u_pred_3d = w1 * x_3d + w2 * v_3d + b
    u_actual_3d = u[start : end - effective_tau]
    u_current_3d = u[start + effective_tau : end]

    min_len = min(len(x_3d), len(u_actual_3d), len(u_current_3d))
    x_3d, v_3d = x_3d[:min_len], v_3d[:min_len]
    u_pred_3d = u_pred_3d[:min_len]
    u_actual_3d = u_actual_3d[:min_len]
    u_current_3d = u_current_3d[:min_len]

    x_range = np.linspace(x_3d.min() - 0.2, x_3d.max() + 0.2, 14)
    v_range = np.linspace(v_3d.min() - 0.2, v_3d.max() + 0.2, 14)
    X_grid, V_grid = np.meshgrid(x_range, v_range)
    U_plane = w1 * X_grid + w2 * V_grid + b

    # z_floor_val = min(u_pred_3d.min(), U_plane.min(), -1.0) - 0.5
    z_floor_val =  -5
    z_floor = np.full_like(x_3d, z_floor_val)

    with fig.batch_update():
        # plane
        fig.data[0].x = X_grid
        fig.data[0].y = V_grid
        fig.data[0].z = U_plane

        #  pred
        fig.data[1].x = x_3d
        fig.data[1].y = v_3d
        fig.data[1].z = u_pred_3d

        # target
        fig.data[2].x = x_3d
        fig.data[2].y = v_3d
        fig.data[2].z = u_actual_3d
        fig.data[2].marker.color = u_actual_3d

        # floor
        fig.data[3].x = x_3d
        fig.data[3].y = v_3d
        fig.data[3].z = z_floor
        fig.data[3].marker.color = u_current_3d

        fig.update_layout(
            scene=dict(
                zaxis=dict(
                    nticks=4,
                    range=[-5, 5],
                ),
            )
        )

        fig.layout.title.text = (
            f"Jupyter Live Compute | R²: {r2:.2f} | Accuracy: {acc:.1f}%"
        )


# Weidgests
mu_slider = widgets.FloatSlider(
    value=13.7, min=0.1, max=30.0, step=0.1, description="Damping (μ):"
)
tau_slider = widgets.FloatSlider(
    value=1.17, min=0.0, max=5.0, step=0.01, description="Delay (τ):"
)
k_slider = widgets.FloatSlider(
    value=2.0, min=0.5, max=15.0, step=0.1, description="Stiffness (k):"
)

out = widgets.interactive_output(
    update_plot, {"mu": mu_slider, "tau": tau_slider, "k": k_slider}
)

widgets.VBox([widgets.HBox([mu_slider, tau_slider, k_slider]), fig])

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

In [37]:
time = 500
dt = 0.01
steps = int(time / dt)
test_size = 0.2
m = 1.0
start, end = 4000, 5000

t = np.linspace(0, time, steps)
u = np.sin(t) / (2 * np.pi)


@njit
def run_simulation(steps, dt, m, k, mu, u):
    x, v = np.zeros(steps), np.zeros(steps)
    x[0], v[0] = 1.0, 0.0
    for i in range(1, steps):
        x_dot = v[i - 1]
        v_dot = -mu / m * v[i - 1] - k / m * x[i - 1] + u[i - 1] / m
        x[i] = x[i - 1] + x_dot * dt
        v[i] = v[i - 1] + v_dot * dt
    return x, v


fig = go.FigureWidget()  # We use this to update

fig.add_trace(
    go.Surface(colorscale="YlOrRd", opacity=0.15, showscale=False, name="Plane")
)
fig.add_trace(
    go.Scatter3d(mode="lines", line=dict(color="blue", width=6), name="Prediction")
)
fig.add_trace(
    go.Scatter3d(
        mode="lines+markers",
        marker=dict(
            size=3.5,
            colorscale="Cividis",
            cmin=-1,
            cmax=1,
        ),
        line=dict(color="rgba(100,100,100,0.3)", width=2, dash="dash"),
        name="Target",
    )
)
fig.add_trace(
    go.Scatter3d(
        mode="markers",
        marker=dict(size=4.5, colorscale="Electric"),
        name="Drive (τ=0)",
    )
)
fig.update_layout(
    template="plotly_white",
    width=900,
    height=700,
    margin=dict(l=0, r=0, b=0, t=40),
    scene=dict(
        xaxis_title="Position (x)",
        yaxis_title="Velocity (v)",
        zaxis_title="Force Scales (u)",
        camera=dict(eye=dict(x=1.4, y=1.4, z=1.2)),
    ),
)


def update_plot(mu, tau, k):
    tau_steps = int(tau / dt)

    # sim
    x, v = run_simulation(steps, dt, m, k, mu, u)
    X = np.column_stack((x, v))

    # model
    effective_tau = max(1, tau_steps)
    X_train, X_test, y_train, y_test = train_test_split(
        X[effective_tau:],
        u[:-effective_tau],
        test_size=test_size,
        random_state=42,
    )
    ridge_model = RidgeCV().fit(X_train, y_train)

    w1, w2 = ridge_model.coef_[0], ridge_model.coef_[1]
    b = ridge_model.intercept_

    # metric
    y_pred = ridge_model.predict(X_test)
    r2 = r2_score(y_test, y_pred)

    # where we slice time
    x_3d = x[start + effective_tau : end]
    v_3d = v[start + effective_tau : end]
    u_pred_3d = w1 * x_3d + w2 * v_3d + b
    u_actual_3d = u[start : end - effective_tau]
    u_current_3d = u[start + effective_tau : end]

    min_len = min(len(x_3d), len(u_actual_3d), len(u_current_3d))
    x_3d, v_3d = x_3d[:min_len], v_3d[:min_len]
    u_pred_3d = u_pred_3d[:min_len]
    u_actual_3d = u_actual_3d[:min_len]
    u_current_3d = u_current_3d[:min_len]

    x_range = np.linspace(x_3d.min() - 0.2, x_3d.max() + 0.2, 14)
    v_range = np.linspace(v_3d.min() - 0.2, v_3d.max() + 0.2, 14)
    X_grid, V_grid = np.meshgrid(x_range, v_range)
    U_plane = w1 * X_grid + w2 * V_grid + b

    z_floor_val = min(u_pred_3d.min(), U_plane.min(), -1.0) - 0.5
    z_floor = np.full_like(x_3d, z_floor_val)

    with fig.batch_update():
        # plane
        fig.data[0].x = X_grid
        fig.data[0].y = V_grid
        fig.data[0].z = U_plane

        #  pred
        fig.data[1].x = x_3d
        fig.data[1].y = v_3d
        fig.data[1].z = u_pred_3d

        # # target
        fig.data[2].x = x_3d
        fig.data[2].y = v_3d
        fig.data[2].z = u_actual_3d
        fig.data[2].marker.color = u_actual_3d

        # floor
        fig.data[3].x = x_3d
        fig.data[3].y = v_3d
        fig.data[3].z = z_floor
        fig.data[3].marker.color = u_current_3d

        fig.layout.title.text = (
            f"Jupyter Live Compute | R²: {r2:.2f}"
        )


# Weidgests
mu_slider = widgets.FloatSlider(
    value=13.7, min=0.1, max=30.0, step=0.1, description="Damping (μ):"
)
tau_slider = widgets.FloatSlider(
    value=1.17, min=0.0, max=5.0, step=0.01, description="Delay (τ):"
)
k_slider = widgets.FloatSlider(
    value=2.0, min=0.5, max=15.0, step=0.1, description="Stiffness (k):"
)

out = widgets.interactive_output(
    update_plot, {"mu": mu_slider, "tau": tau_slider, "k": k_slider}
)

widgets.VBox([widgets.HBox([mu_slider, tau_slider, k_slider]), fig])

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

...

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()